In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_annotated/checkpoints/pre_cell_0.pickle

In [4]:
%%RecordEvent
%%time
### cell 0 ###

### cell 0 – optimized for cudf
from pathlib import Path
import os, re

base_input = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input"
mobile_list = []
broadband_list = []

# precompile regex to extract year and quarter
pattern = re.compile(r"year_(\d+)_quarter_(\d+)\.csv")
for filepath in base_input.rglob("*.csv"):
    # read and convert to nullable dtypes on GPU
    df = pd.read_csv(filepath, thousands=',').convert_dtypes()
    # rename columns if needed
    df.rename(columns={'Number of Record': 'Number of Records'}, inplace=True)
    # extract year and quarter
    yr, qt = map(int, pattern.search(filepath.name).groups())
    df['Year'] = np.int64(yr)
    df['Quarter'] = np.int64(qt)
    # collect per category
    if 'mobile' in filepath.name:
        mobile_list.append(df)
    else:
        broadband_list.append(df)

# single concat per category on GPU
Mobile_df = pd.concat(mobile_list, ignore_index=True)
Broadband_df = pd.concat(broadband_list, ignore_index=True)

# ensure Year/Quarter are int64 and sort
Mobile_df = (Mobile_df
    .astype({'Year': np.int64, 'Quarter': np.int64})
    .sort_values(['Year', 'Quarter'])
)
Broadband_df = (Broadband_df
    .astype({'Year': np.int64, 'Quarter': np.int64})
    .sort_values(['Year', 'Quarter'])
)

# expand datasets
factor = 10
Mobile_df = pd.concat([Mobile_df] * (factor * 2), ignore_index=True)
Broadband_df = pd.concat([Broadband_df] * factor, ignore_index=True)

print(Broadband_df.shape)
print(Mobile_df.shape)
Mobile_df.info()
Broadband_df.info()

(25970, 13)
(49740, 13)
<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 49740 entries, 0 to 49739
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               49740 non-null  string
 1   Number of Records  49740 non-null  Int64
 2   Devices            49740 non-null  Int64
 3   Tests              49740 non-null  Int64
 4   Avg. Avg U Kbps    49740 non-null  Int64
 5   Avg. Avg D Kbps    49740 non-null  Int64
 6   Avg Lat Ms         49740 non-null  Int64
 7   Avg. Pop2005       49740 non-null  Int64
 8   Rank Upload        49740 non-null  Int64
 9   Rank Download      49740 non-null  Int64
 10  Rank Latency       49740 non-null  Int64
 11  Year               49740 non-null  int64
 12  Quarter            49740 non-null  int64
dtypes: Int64(10), int64(2), string(1)
memory usage: 5.2 MB
<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 25970 entries, 0 to 25969
Data columns (total 13 columns):
 # 

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_0_try_4.pickle